# НИРМ 4 — метод с учётом пола: обучение вариантов CNN-LSTM (Google Colab)

Проверка предложенной новизны: сокращается ли разрыв качества распознавания между мужчинами и женщинами,
если (1) дать модели возраст и пол как клинические признаки и (2) уравновесить вклад полов при обучении.
Анализ — локально, в `nirm4_sexaware_ptbxl.ipynb`.

Четыре варианта CNN-LSTM (архитектура и гиперпараметры недели 8, lr 1e-3), по 5 seed каждый:
- `base` — как на неделе 8 (контроль; seed 42 должен повторить val macro-AUC 0.9363);
- `demo` — + возраст и пол: небольшой слой 2 → 16 поверх, конкатенация с выходом attention-pooling;
- `groupw` — веса примеров в функции потерь: внутри каждого класса положительные и отрицательные примеры
  мужчин и женщин дают равный суммарный вклад;
- `demo_groupw` — оба изменения.

**Перед запуском** в папке `thesis_ptbxl/` на Google Drive должны лежать `X100_bp05-40_crop.npy` (уже
загружен) и новый `colab_meta_v2.npz` из `notebooks/cache/` (добавлен возраст). Затем T4 GPU и `Run all`;
20 запусков, около 30 минут. При обрыве сессии — снова `Run all`, обучение продолжится.

In [ ]:
import json
import os
import sys
import time

import numpy as np
import torch
import torch.nn as nn
from sklearn.metrics import roc_auc_score

IN_COLAB = 'google.colab' in sys.modules
SMOKE = os.environ.get('SMOKE') == '1'

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    CANDIDATES = ['/content/drive/MyDrive/thesis_ptbxl/', '/content/drive/MyDrive/Colab Notebooks/thesis_ptbxl/']
    ROOT = next((d for d in CANDIDATES if os.path.exists(d + 'colab_meta_v2.npz')), None)
    assert ROOT, f'не найден colab_meta_v2.npz ни в одной из папок: {CANDIDATES}'
else:
    ROOT = 'cache/'
OUT = os.environ.get('COLAB_OUT', ROOT)
RESULTS, CKPT = OUT + 'results_nirm4/', OUT + 'ckpt_nirm4/'
os.makedirs(RESULTS, exist_ok=True)
os.makedirs(CKPT, exist_ok=True)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
ENV = {'torch': torch.__version__, 'cuda': torch.version.cuda, 'device': DEVICE,
       'gpu': torch.cuda.get_device_name(0) if DEVICE == 'cuda' else None}
print(ENV)

In [ ]:
meta = np.load(ROOT + 'colab_meta_v2.npz')
X_all = np.load(ROOT + 'X100_bp05-40_crop.npy', mmap_mode='r')
Y_all, split, has_label, sex = meta['Y'].astype(np.float32), meta['split'], meta['has_label'], meta['sex']
LEAD_MEAN, LEAD_STD = meta['lead_mean'].astype(np.float32), meta['lead_std'].astype(np.float32)
FS = int(meta['fs'])
idx = {s: np.where((split == s) & has_label)[0] for s in ('train', 'val', 'test')}
if SMOKE:
    idx = {s: v[:{'train': 600, 'val': 300, 'test': 300}[s]] for s, v in idx.items()}

# возраст нормируется статистикой обучающей выборки; пол: женщины 1, мужчины 0
age = meta['age'].astype(np.float32)
AGE_MEAN, AGE_STD = age[idx['train']].mean(), age[idx['train']].std()
DEMO_ALL = np.stack([(age - AGE_MEAN) / AGE_STD, (sex == 'female').astype(np.float32)], axis=1)


def group_weights(rows):
    """Вес (запись, класс): внутри класса положительные и отрицательные примеры каждого пола в сумме равны."""
    y, female = Y_all[rows], sex[rows] == 'female'
    w = np.ones_like(y)
    for k in range(y.shape[1]):
        for label in (0, 1):
            n_label = (y[:, k] == label).sum()
            for grp in (female, ~female):
                m = grp & (y[:, k] == label)
                w[m, k] = n_label / (2 * m.sum())
    return w


def to_device(rows):
    x = (np.asarray(X_all[rows]) - LEAD_MEAN) / LEAD_STD
    return torch.from_numpy(x.astype(np.float32)).to(DEVICE)


DATA = {s: (to_device(r), torch.from_numpy(Y_all[r]).to(DEVICE), torch.from_numpy(DEMO_ALL[r]).to(DEVICE))
        for s, r in idx.items()}
W_TRAIN = torch.from_numpy(group_weights(idx['train'])).to(DEVICE)
print({s: tuple(x.shape) for s, (x, _, _) in DATA.items()},
      f'веса: женщины с MI {W_TRAIN[(DATA["train"][2][:, 1] == 1) & (DATA["train"][1][:, 1] == 1), 1][0]:.2f}, '
      f'мужчины с MI {W_TRAIN[(DATA["train"][2][:, 1] == 0) & (DATA["train"][1][:, 1] == 1), 1][0]:.2f}')

## Модель (как на неделе 8, плюс необязательный вход клинических признаков)

In [ ]:
def conv_block(c_in, c_out, k=7, pool=True):
    layers = [nn.Conv1d(c_in, c_out, k, padding='same', bias=False), nn.BatchNorm1d(c_out), nn.ReLU(),
              nn.Conv1d(c_out, c_out, k, padding='same', bias=False), nn.BatchNorm1d(c_out), nn.ReLU()]
    return nn.Sequential(*layers, nn.MaxPool1d(2)) if pool else nn.Sequential(*layers)


class AttentionPool(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.score = nn.Sequential(nn.Linear(d, d // 2), nn.Tanh(), nn.Linear(d // 2, 1))
        self.weights = None

    def forward(self, h):
        w = torch.softmax(self.score(h).squeeze(-1), dim=1)
        self.weights = w.detach()
        return (w.unsqueeze(-1) * h).sum(dim=1)


def conv_stem(width, n_leads=12):
    return nn.Sequential(conv_block(n_leads, width // 2), conv_block(width // 2, width))


class CNNLSTM(nn.Module):
    """use_demo=False — ровно модель недели 8 (тот же порядок создания слоёв, те же веса при том же seed)."""

    def __init__(self, width=128, hidden=128, n_classes=5, use_demo=False, demo_dim=16):
        super().__init__()
        self.stem = conv_stem(width)
        self.lstm = nn.LSTM(width, hidden, batch_first=True, bidirectional=True)
        self.pool = AttentionPool(2 * hidden)
        self.head = nn.Sequential(nn.Dropout(0.5), nn.Linear(2 * hidden + (demo_dim if use_demo else 0), n_classes))
        self.demo = nn.Sequential(nn.Linear(2, demo_dim), nn.ReLU()) if use_demo else None

    def forward(self, x, demo=None):
        h, _ = self.lstm(self.stem(x).transpose(1, 2))
        z = self.pool(h)
        if self.demo is not None:
            z = torch.cat([z, self.demo(demo)], dim=1)
        return self.head(z)


VARIANTS = {'base': (False, False), 'demo': (True, False), 'groupw': (False, True), 'demo_groupw': (True, True)}

## Обучение (протокол недели 8: 20 эпох, AdamW, one-cycle, батч 128, выбор эпохи по фолду 9)

In [ ]:
EPOCHS = 1 if SMOKE else 20
BATCH = 128
WD = 1e-2
LR = 1e-3
WIN = int(2.5 * FS)


def macro_auc(y, s):
    return float(np.mean([roc_auc_score(y[:, k], s[:, k]) for k in range(y.shape[1])]))


@torch.no_grad()
def predict(model, x, demo, batch=1024):
    model.eval()
    w = x.unfold(1, WIN, FS)
    n, n_win = w.shape[:2]
    flat = w.reshape(n * n_win, w.shape[2], WIN)
    d = demo.repeat_interleave(n_win, dim=0)
    out = torch.cat([torch.sigmoid(model(flat[s:s + batch], d[s:s + batch])) for s in range(0, len(flat), batch)])
    return out.reshape(n, n_win, -1).mean(dim=1).cpu().numpy()


def save_checkpoint(path, model, opt, sched, epoch, history, best, rng):
    torch.save({'model': model.state_dict(), 'opt': opt.state_dict(), 'sched': sched.state_dict(),
                'epoch': epoch, 'history': history, 'best': best, 'np_rng': rng.bit_generator.state,
                'torch_rng': torch.get_rng_state(),
                'cuda_rng': torch.cuda.get_rng_state_all() if DEVICE == 'cuda' else None}, path)


def restore_checkpoint(path, model, opt, sched, rng):
    ck = torch.load(path, map_location=DEVICE, weights_only=False)
    model.load_state_dict(ck['model'])
    opt.load_state_dict(ck['opt'])
    sched.load_state_dict(ck['sched'])
    rng.bit_generator.state = ck['np_rng']
    torch.set_rng_state(ck['torch_rng'])
    if ck['cuda_rng'] is not None:
        torch.cuda.set_rng_state_all(ck['cuda_rng'])
    return ck['epoch'], ck['history'], ck['best']


def train_epoch(model, opt, sched, rng, weighted):
    x_tr, y_tr, d_tr = DATA['train']
    model.train()
    order, losses = rng.permutation(len(x_tr)), []
    offsets = torch.arange(WIN, device=DEVICE)
    bce = nn.BCEWithLogitsLoss()
    bce_each = nn.BCEWithLogitsLoss(reduction='none')
    for s in range(0, len(order), BATCH):
        rows = torch.from_numpy(order[s:s + BATCH]).to(DEVICE)
        start = torch.from_numpy(rng.integers(0, x_tr.shape[1] - WIN + 1, len(rows))).to(DEVICE)
        xb = x_tr[rows[:, None], start[:, None] + offsets].permute(0, 2, 1)
        logits = model(xb, d_tr[rows])
        loss = (bce_each(logits, y_tr[rows]) * W_TRAIN[rows]).mean() if weighted else bce(logits, y_tr[rows])
        opt.zero_grad()
        loss.backward()
        opt.step()
        sched.step()
        losses.append(loss.item())
    return float(np.mean(losses))


def run(variant, seed):
    rid = f'cnnlstm_{variant}_s{seed}' + ('_smoke' if SMOKE else '')
    if os.path.exists(RESULTS + rid + '.json'):
        return json.load(open(RESULTS + rid + '.json'))
    use_demo, weighted = VARIANTS[variant]
    torch.manual_seed(seed)
    rng = np.random.default_rng(seed)
    model = CNNLSTM(use_demo=use_demo).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)
    steps = int(np.ceil(len(DATA['train'][0]) / BATCH))
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=LR, epochs=EPOCHS, steps_per_epoch=steps)
    start_epoch, history, best = 0, [], {'auc': -1.0, 'epoch': 0, 'state': None}
    if os.path.exists(CKPT + rid + '.pt'):
        start_epoch, history, best = restore_checkpoint(CKPT + rid + '.pt', model, opt, sched, rng)
        print(f'{rid}: продолжение с эпохи {start_epoch + 1}')
    x_va, y_va, d_va = DATA['val']
    for epoch in range(start_epoch, EPOCHS):
        t0 = time.perf_counter()
        loss = train_epoch(model, opt, sched, rng, weighted)
        val_auc = macro_auc(y_va.cpu().numpy(), predict(model, x_va, d_va))
        history.append({'epoch': epoch + 1, 'train_loss': loss, 'val_macro_auc': val_auc,
                        'sec': time.perf_counter() - t0})
        if val_auc > best['auc']:
            best = {'auc': val_auc, 'epoch': epoch + 1,
                    'state': {k: v.detach().clone() for k, v in model.state_dict().items()}}
        save_checkpoint(CKPT + rid + '.pt', model, opt, sched, epoch + 1, history, best, rng)
    model.load_state_dict(best['state'])
    s_val = predict(model, *DATA['val'][::2])
    s_test = predict(model, *DATA['test'][::2])
    np.savez(RESULTS + rid + '.npz', s_val=s_val, s_test=s_test, val_idx=idx['val'], test_idx=idx['test'])
    torch.save(best['state'], RESULTS + rid + '_weights.pt')
    info = {'run_id': rid, 'variant': variant, 'seed': seed, 'use_demo': use_demo, 'weighted': weighted,
            'best_epoch': best['epoch'], 'val_macro_auc': best['auc'],
            'n_params': sum(p.numel() for p in model.parameters()), 'history': history,
            'train_minutes': sum(h['sec'] for h in history) / 60, 'env': ENV, 'smoke': SMOKE,
            'age_mean': float(AGE_MEAN), 'age_std': float(AGE_STD)}
    json.dump(info, open(RESULTS + rid + '.json', 'w'), indent=1)
    os.remove(CKPT + rid + '.pt')
    print(f"{rid}: эпоха {best['epoch']}, val macro-AUC {best['auc']:.4f}, {info['train_minutes']:.1f} мин")
    return info

In [ ]:
SEEDS = (42,) if SMOKE else (42, 1, 2, 3, 4)
for variant in VARIANTS:
    for seed in SEEDS:
        run(variant, seed)

## Сводка

Контроль: `cnnlstm_base_s42` должен дать val macro-AUC 0.9363, как `cnn_lstm_w2.5_c128_lr0.001_s42` недели 8.
Скачайте папку `thesis_ptbxl/results_nirm4/` в `notebooks/cache/results_nirm4/`.

In [ ]:
rows = [json.load(open(RESULTS + f)) for f in sorted(os.listdir(RESULTS)) if f.endswith('.json')]
for v in VARIANTS:
    vals = [r['val_macro_auc'] for r in rows if r['variant'] == v]
    print(f'{v:<12} val macro-AUC по seed: ' + ' '.join(f'{a:.4f}' for a in vals)
          + (f'  (среднее {np.mean(vals):.4f})' if vals else ''))